# Day 04 — File I/O · JSON
**Module 01: Python + Async + FastAPI for LLM Engineering**  
Vidya Sankalp | Applied GenAI Engineering

> **What you'll learn:** `with open()` · file modes `r` `w` `a` · `json.loads()` · `json.dumps()` · `json.load()` · `json.dump()` · `csv.DictReader` · JSONL logging

> **Connection to previous days:** f-strings (Day 01) fill the prompt templates loaded from files today. Dicts (Day 01/03) store the parsed LLM responses.


---


## 1 — Imports


In [2]:
import json           # parse and produce JSON strings
import csv            # read CSV files row by row
import io             # io.StringIO — treat a string like a file
from pathlib import Path   # cleaner file path handling

# __file__ only exists when running as a .py script.
# In Jupyter notebooks it is not defined → use Path.cwd() instead.
try:
    BASE_DIR = Path(__file__).parent.parent
except NameError:
    BASE_DIR = Path.cwd()   # Jupyter: current working directory

PROMPT_DIR = BASE_DIR / 'prompts'

print('BASE_DIR  :', BASE_DIR)
print('PROMPT_DIR:', PROMPT_DIR)


BASE_DIR  : /Users/akellaprudhvi/PycharmProjects/vs-python/notebooks
PROMPT_DIR: /Users/akellaprudhvi/PycharmProjects/vs-python/notebooks/prompts


---


## 2 — JSON — `json.loads()` and `json.dumps()`

| Function | Direction | When to use |
|----------|-----------|-------------|
| `json.loads(string)` | string → Python dict | Parse LLM response |
| `json.dumps(dict)` | Python dict → string | Write log entries |
| `json.load(file)` | open file → Python dict | Load a `.json` config file |
| `json.dump(dict, file)` | Python dict → open file | Save results to `.json` |

> **The LLM API always returns a string** — even when the LLM was told to return JSON.  
> You must call `json.loads()` before you can access any field.


In [3]:
# ── json.loads() — string → dict ───────────────────────────
raw = '{"category": "TRACK_ORDER", "confidence": "high"}'

print('Before:', type(raw), raw)
parsed = json.loads(raw)
print('After :', type(parsed), parsed)
print('Access:', parsed['category'])


Before: <class 'str'> {"category": "TRACK_ORDER", "confidence": "high"}
After : <class 'dict'> {'category': 'TRACK_ORDER', 'confidence': 'high'}
Access: TRACK_ORDER


In [4]:
# ── json.dumps() — dict → string ───────────────────────────
config = {'model': 'gpt-4o', 'max_tokens': 1024, 'temperature': 0.2}

compact = json.dumps(config)           # single line — for logs
pretty  = json.dumps(config, indent=2) # indented — for .json files

print('Compact:', compact)
print('Type   :', type(compact))   # str
print()
print('Pretty:')
print(pretty)


Compact: {"model": "gpt-4o", "max_tokens": 1024, "temperature": 0.2}
Type   : <class 'str'>

Pretty:
{
  "model": "gpt-4o",
  "max_tokens": 1024,
  "temperature": 0.2
}


In [12]:
# json.dump  → write a dict directly to an open file
with open('model_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

In [6]:
# ── json.load() and json.dump() — file versions ─────────────

# json.load  → read a dict directly from an open file

with open('model_config.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print('Loaded back:', loaded)
print('Type       :', type(loaded).__name__)   # dict


Loaded back: {'model': 'gpt-4o', 'max_tokens': 1024, 'temperature': 0.2}
Type       : dict


---


## 3 — The `with` Statement — Safe File Handling

The `with` statement guarantees the file is **closed automatically** — even if an error occurs inside the block.

```python
# Without with — risky:
f    = open('file.txt')
data = f.read()    # if this raises an error, f.close() is NEVER called
f.close()

# With the with statement — safe:
with open('file.txt') as f:
    data = f.read()    # file is always closed when the block exits
```

| Mode | String | Behaviour |
|------|--------|-----------|
| Read | `'r'` | Open existing file. `FileNotFoundError` if missing. |
| Write | `'w'` | Create file. **Overwrites** if already exists. |
| Append | `'a'` | Add to the end. Creates file if missing. |

> Always add `encoding='utf-8'` to avoid silent text corruption on Windows.


### Reading a file — `load_system_prompt()`


In [15]:
def load_system_prompt(filepath='sample_prompt.txt'):
    """
    Load a system prompt from a .txt file.
    Keeping prompts in files means the team can edit them without touching Python.
    """
    # 'r' = read mode  |  encoding='utf-8' = always specify
    # with block closes the file automatically — even if f.read() raises an error
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()   # loads the entire file as one string

    # .strip() removes the trailing newline that text editors always add
    return content.strip()


In [13]:
# Write a prompt to a file — simulates prompts/system_prompt.txt
with open('sample_prompt.txt', 'w', encoding='utf-8') as f:
    f.write('## Role\n')
    f.write('You are a customer support assistant for ShopSmart.\n')
    f.write('\n')
    f.write('## Task\n')
    f.write('Answer using ONLY the information provided.')

In [16]:
# Now call the function — reads the file back
prompt = load_system_prompt('sample_prompt.txt')
print(f'Loaded {len(prompt)} characters:')
print(prompt)

Loaded 112 characters:
## Role
You are a customer support assistant for ShopSmart.

## Task
Answer using ONLY the information provided.


### Appending to a file — `save_llm_response_log()` — uses `json.dumps()`


In [17]:
def save_llm_response_log(query, response, log_file='llm_calls.log'):
    """
    Append one LLM call to a JSONL log file.
    JSONL = JSON Lines: one JSON object per line.
    """
    # 'a' = append mode:
    #   file exists  → adds to the END, does not overwrite previous entries
    #   file missing → creates it automatically
    with open(log_file, 'a', encoding='utf-8') as f:
        # json.dumps() converts a Python dict → JSON string
        entry = json.dumps({'query': query, 'response': response})
        f.write(entry + '\n')   # one entry per line = JSONL format


In [ ]:
# Call the function twice — 'a' mode appends each call to the same file
# TRACE:
# Call 1 → file missing → 'a' creates llm_calls.log → writes line 1
# Call 2 → file exists  → 'a' appends              → writes line 2 (line 1 untouched)

In [18]:
save_llm_response_log('Where is my order?', 'Your order ORD-3042 is In Transit.')

In [19]:
save_llm_response_log('Can I return this?', 'Returns accepted within 30 days.')

In [20]:
# Read back every line — same filename the function wrote to
# TRACE:
# line 0 → '{"query": "Where is my order?", "response": "..."}\n'
#           .strip() removes \n  →  json.loads() converts string → dict
# line 1 → second entry, same process
print('Log contents:')
with open('llm_calls.log', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        record = json.loads(line.strip())
        print(f'  [{i+1}] Q: {record["query"]}')
        print(f'       A: {record["response"]}')

Log contents:
  [1] Q: Where is my order?
       A: Your order ORD-3042 is In Transit.
  [2] Q: Can I return this?
       A: Returns accepted within 30 days.


---


### `parse_llm_json_response()` — handles plain and fenced JSON


In [38]:
def parse_llm_json_response(raw_response):
    """
    Parse a JSON string from the LLM into a Python dict.
    Also handles responses wrapped in markdown fences:
        ```json
        {"key": "value"}
        ```
    """
    # Step 1: remove leading/trailing whitespace
    cleaned = raw_response.strip()

    # Step 2: strip markdown fences if present
    # LLMs often add ```json ... ``` even when told not to
    if cleaned.startswith('```'):
        # split('\n', 1) → split on first newline only
        # [-1] takes everything after the opening fence line
        cleaned = cleaned.split('\n', 1)[-1]
        # rsplit('```', 1) → split on last ``` only
        # [0] takes everything before the closing fence
        cleaned = cleaned.rsplit('```', 1)[0].strip()

    # Step 3: convert JSON string → Python dict
    return json.loads(cleaned)


In [39]:
# ── Demo 1: plain JSON string ────────────────────────────────
raw = '{"category": "TRACK_ORDER", "confidence": "high"}'

print('Before parse — type:', type(raw).__name__)   # str
parsed = parse_llm_json_response(raw)
print('After parse  — type:', type(parsed).__name__) # dict
print('category  :', parsed['category'])
print('confidence:', parsed['confidence'])


Before parse — type: str
After parse  — type: dict
category  : TRACK_ORDER
confidence: high


In [40]:
# ── Demo 2: LLM wrapped response in markdown fences ──────────
# This is very common — LLMs add ```json even when told not to
fenced = '```json\n{"intent": "CANCEL", "urgency": "high"}\n```'

print('Raw response from LLM:')
print(fenced)
print()

# TRACE:
# cleaned = fenced.strip()
# startswith('```') → True
# split('\n',1)[-1]  → '{"intent": "CANCEL", "urgency": "high"}\n```'
# rsplit('```',1)[0]  → '{"intent": "CANCEL", "urgency": "high"}\n'
# .strip()            → '{"intent": "CANCEL", "urgency": "high"}'
# json.loads()        → {'intent': 'CANCEL', 'urgency': 'high'}

parsed2 = parse_llm_json_response(fenced)
print('After parse:', parsed2)
print('intent    :', parsed2['intent'])


Raw response from LLM:
```json
{"intent": "CANCEL", "urgency": "high"}
```

After parse: {'intent': 'CANCEL', 'urgency': 'high'}
intent    : CANCEL


---


## 4 — Reading CSV Data with `csv.DictReader`

`csv.DictReader` reads the first row as column headers and maps every subsequent row to those headers.  
Each row becomes a dict — exactly the same shape as a database row in Python.

> **Important:** every CSV value is a **string**.  
> Always convert: `int(row['customer_id'])`, `float(row['price'])`.


### Reading customer rows from CSV


In [54]:
import csv
# sample_csv simulates what customers.csv looks like on disk
# io.StringIO wraps a string so csv.DictReader treats it like an open file
with open('employee.csv', 'r', encoding='utf-8') as f:
    # csv.DictReader reads row 1 as headers, maps each subsequent row to them
    # ITERATION TRACE:
    # Row 1 → {'customer_id':'1001', 'name':'Prudhvi Akella', 'city':'Rajahmundry', ...}
    # Row 2 → {'customer_id':'1002', 'name':'Ravi Kumar',     'city':'Hyderabad', ...}
    # Row 3 → {'customer_id':'1003', 'name':'Ananya Sharma',  'city':'Bangalore', ...}
    # EOF   → loop ends

    reader    = csv.DictReader(f)
    customers = list(reader)
    print(customers)

[{'customer_id': '1001', 'name': 'Prudhvi Akella', 'email': 'prudhvi@example.com', 'city': 'Rajahmundry', 'state': 'AP'}, {'customer_id': '1002', 'name': 'Ravi Kumar', 'email': 'ravi@example.com', 'city': 'Hyderabad', 'state': 'TS'}, {'customer_id': '1003', 'name': 'Ananya Sharma', 'email': 'ananya@example.com', 'city': 'Bangalore', 'state': 'KA'}]


In [55]:
for c in customers:
    print(f"  {c['customer_id']}  {c['name']:20s}  {c['city']}, {c['state']}")

print()
print('--- CSV type trap ---')
print('customer_id type    :', type(customers[0]['customer_id']).__name__, '← str!')
print("String compare trap : '1001' > '200' =", '1001' > '200', '← WRONG (alphabetic)')
print('Correct int compare : 1001  > 200  =',   1001  > 200)
print('Fix                 :', int(customers[0]['customer_id']))


  1001  Prudhvi Akella        Rajahmundry, AP
  1002  Ravi Kumar            Hyderabad, TS
  1003  Ananya Sharma         Bangalore, KA

--- CSV type trap ---
customer_id type    : str ← str!
String compare trap : '1001' > '200' = False ← WRONG (alphabetic)
Correct int compare : 1001  > 200  = True
Fix                 : 1001


### What if the delimiter is not a comma?

Not all CSV files use commas. The `delimiter` parameter tells `DictReader` what character separates the columns.

| File type | Delimiter | `delimiter=` |
|-----------|-----------|-------------|
| Standard CSV | `,` comma | default — no need to specify |
| European CSV | `;` semicolon | `delimiter=';'` |
| TSV (Tab Separated) | `\t` tab | `delimiter='\t'` |
| Pipe separated | `\|` pipe | `delimiter='\|'` |


In [56]:
# ── Semicolon-delimited (common in European exports) ────────
semicolon_csv = """customer_id;name;city
1001;Prudhvi Akella;Rajahmundry
1002;Ravi Kumar;Hyderabad
"""

reader = csv.DictReader(io.StringIO(semicolon_csv), delimiter=';')
for row in reader:
    print(row)


{'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
{'customer_id': '1002', 'name': 'Ravi Kumar', 'city': 'Hyderabad'}


In [57]:
# ── Tab-separated / TSV ─────────────────────────────────────
tsv_data = 'customer_id\tname\tcity\n1001\tPrudhvi Akella\tRajahmundry\n1002\tRavi Kumar\tHyderabad'

reader = csv.DictReader(io.StringIO(tsv_data), delimiter='\t')
for row in reader:
    print(row)


{'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
{'customer_id': '1002', 'name': 'Ravi Kumar', 'city': 'Hyderabad'}


In [58]:
# ── Pipe-separated ──────────────────────────────────────────
pipe_csv = """customer_id|name|city
1001|Prudhvi Akella|Rajahmundry
1002|Ravi Kumar|Hyderabad
"""

reader = csv.DictReader(io.StringIO(pipe_csv), delimiter='|')
for row in reader:
    print(row)


{'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
{'customer_id': '1002', 'name': 'Ravi Kumar', 'city': 'Hyderabad'}


### What if the file has no header row?

Use the `fieldnames` parameter to supply column names manually.


In [59]:
# No header row — supply column names with fieldnames=
no_header_csv = """1001,Prudhvi Akella,Rajahmundry
1002,Ravi Kumar,Hyderabad
"""

reader = csv.DictReader(
    io.StringIO(no_header_csv),
    fieldnames=['customer_id', 'name', 'city']   # supply headers manually
)
for row in reader:
    print(row)


{'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
{'customer_id': '1002', 'name': 'Ravi Kumar', 'city': 'Hyderabad'}


### What if you don't know the delimiter?

`csv.Sniffer` can detect the delimiter automatically by sampling the first few lines of the file.


In [76]:
# csv.Sniffer — auto-detect delimiter
sample = """customer_id,name,email,city,state
1001,Prudhvi Akella,"prudhvi@example.com,prudhvi1@example2.com",Rajahmundry,AP
1002,Ravi Kumar,ravi@example.com,Hyderabad,TS
1003,Ananya Sharma,ananya@example.com,Bangalore,KA
"""

dialect  = csv.Sniffer().sniff(sample)   # sniff() reads a sample and guesses
print('Detected delimiter:', repr(dialect.delimiter))
print("Quote character:", repr(dialect.quotechar))
print(dialect)

Detected delimiter: ','
Quote character: '"'
<class 'csv.Sniffer.sniff.<locals>.dialect'>


In [75]:
# Use the detected dialect in DictReader
reader = csv.DictReader(io.StringIO(sample), dialect=dialect)
for row in reader:
    print(row)

{'customer_id': '1001', 'name': 'Prudhvi Akella', 'email': 'prudhvi@example.com,prudhvi1@example2.com', 'city': 'Rajahmundry', 'state': 'AP'}
{'customer_id': '1002', 'name': 'Ravi Kumar', 'email': 'ravi@example.com', 'city': 'Hyderabad', 'state': 'TS', None: ['1002']}
{'customer_id': '1003', 'name': 'Ananya Sharma', 'email': 'ananya@example.com', 'city': 'Bangalore', 'state': 'KA'}


---


## 5 — Full Pipeline — step by step

Each step uses a skill from this notebook. Run them in order — variables carry forward.


### Step 1 — Load the system prompt from a file


In [ ]:
system_prompt = load_system_prompt('sample_prompt.txt')

print(f'Loaded {len(system_prompt)} chars')
print(system_prompt)


### Step 2 — Read customer data


In [ ]:
# In production this row comes from csv.DictReader (Section 4)
csv_row       = {'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
customer_id   = int(csv_row['customer_id'])   # CSV always gives str → convert
customer_name = csv_row['name']

print(f'Customer: {customer_name}  id={customer_id}  type={type(customer_id).__name__}')


### Step 3 — Build the user message with an f-string


In [ ]:
user_message = f'Customer: {customer_name}\nQuery: Where is my order #3042?'

print(user_message)


### Step 4 — Receive LLM response (simulated with fences)


In [ ]:
# In production this comes back from openai.chat.completions.create()
mock_llm_output = '```json\n{"category": "TRACK_ORDER", "reply": "Your order ORD-3042 is In Transit.", "confidence": "high"}\n```'

print('Raw LLM output:')
print(mock_llm_output)


### Step 5 — Parse the response


In [ ]:
parsed     = parse_llm_json_response(mock_llm_output)
category   = parsed.get('category',   'UNKNOWN')
reply      = parsed.get('reply',       'No reply')
confidence = parsed.get('confidence',  'unknown')

print('Parsed dict :', parsed)
print('category    :', category)
print('reply       :', reply)
print('confidence  :', confidence)


### Step 6 — Log the call


In [ ]:
save_llm_response_log(user_message, reply)

# Read back to confirm it was written
with open('llm_calls.log', 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f'llm_calls.log now has {len(lines)} entries')
print('Last entry:', lines[-1].strip())


---


## Summary — Day 04

| Concept | Key rule |
|---------|----------|
| `with open(...) as f` | File is always closed — even if an error occurs |
| `__file__` in Jupyter | NameError — use `try/except` and fall back to `Path.cwd()` |
| Mode `'r'` | Read existing file |
| Mode `'w'` | Write — **overwrites** if file exists |
| Mode `'a'` | Append — creates file if missing |
| `encoding='utf-8'` | Always specify — avoids silent corruption on Windows |
| `json.loads(string)` | JSON string → Python dict (parse LLM response) |
| `json.dumps(dict)` | Python dict → JSON string (log entries) |
| `json.load(file)` | Read JSON directly from an open file object |
| Strip markdown fences | `if cleaned.startswith('```')` before `json.loads()` |
| `csv.DictReader` | Each CSV row → dict, column headers = keys |
| `delimiter=';'` | Read semicolon / pipe / tab separated files |
| `fieldnames=[...]` | Supply column names when file has no header row |
| `csv.Sniffer().sniff(sample)` | Auto-detect delimiter from file content |
| CSV values are strings | Always convert: `int(row['id'])`, `float(row['price'])` |
| JSONL | One JSON object per line — append mode log files |

**Next:** Day 05 — Exception Handling + Retry + logging  
`try/except` · `finally` · `raise` · `logging` module · retry with exponential backoff
